In [18]:
import torch

In [19]:
n_cells = 1024
n_genes = 30000
x = torch.rand(n_cells, n_genes)

In [20]:
x

tensor([[8.3994e-01, 4.3457e-01, 1.2091e-01,  ..., 6.6965e-01, 2.3422e-02,
         9.5953e-01],
        [7.3456e-01, 7.9757e-01, 7.0356e-01,  ..., 4.2323e-01, 7.0251e-01,
         4.2329e-01],
        [7.7741e-01, 7.5429e-01, 1.5532e-01,  ..., 3.4323e-01, 5.4567e-01,
         4.5592e-01],
        ...,
        [8.2362e-01, 6.4957e-01, 9.5776e-01,  ..., 5.7794e-01, 2.8125e-01,
         9.2083e-04],
        [9.8522e-01, 4.3932e-02, 1.3495e-01,  ..., 1.3343e-01, 2.0349e-01,
         6.8269e-01],
        [5.7875e-01, 3.8042e-01, 4.2704e-01,  ..., 6.6703e-01, 1.5267e-01,
         5.9395e-01]])

In [21]:
import torch.nn as nn

class WeightedGeneEmbeddingPool(nn.Module):
    def __init__(self, n_genes_in: int, emb_dim: int = 512,
                 use_log1p: bool = True, weight_mode: str = "l1"):
        super().__init__()
        self.emb = nn.Embedding(n_genes_in, emb_dim)
        self.use_log1p = use_log1p
        assert weight_mode in {"l1", "softmax"}
        self.weight_mode = weight_mode

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, Gin) non-negative
        if self.use_log1p:
            x = torch.log1p(x)

        if self.weight_mode == "l1":
            denom = x.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            w = x / denom
        else:
            w = F.softmax(x, dim=-1)

        # (B, Gin) @ (Gin, D) -> (B, D)
        return w @ self.emb.weight

In [22]:
first_layer = WeightedGeneEmbeddingPool(n_genes, 1024, use_log1p=True)

In [23]:
first_layer = first_layer.forward(x)

In [24]:
first_layer

tensor([[-0.0007, -0.0012,  0.0026,  ..., -0.0025, -0.0066, -0.0021],
        [ 0.0058,  0.0019,  0.0056,  ...,  0.0020, -0.0078, -0.0092],
        [-0.0025,  0.0073, -0.0004,  ..., -0.0015, -0.0075, -0.0085],
        ...,
        [ 0.0129,  0.0032,  0.0076,  ...,  0.0008, -0.0050, -0.0055],
        [ 0.0057,  0.0096,  0.0088,  ...,  0.0004, -0.0055, -0.0055],
        [ 0.0030, -0.0037,  0.0062,  ..., -0.0010, -0.0070, -0.0080]],
       grad_fn=<MmBackward0>)

In [25]:
# A simple fully-connected residual block.
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super(ResidualBlock, self).__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.bn = nn.BatchNorm1d(out_features)
        self.relu = nn.ReLU(inplace=False)
        # Adjust skip connection if dimensions differ.
        self.residual = nn.Linear(in_features, out_features) if in_features != out_features else nn.Identity()

    def forward(self, x):
        identity = self.residual(x)
        out = self.linear(x)
        out = self.bn(out)
        out = self.relu(out)
        return out + identity
    

# Encoder with residual blocks and storing skip connections.
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=None):
        super(Encoder, self).__init__()
        if hidden_dims is None:
            self.use_skips = False
            self.model = nn.Sequential(
                nn.Linear(input_dim, latent_dim * 2),
                nn.ReLU(inplace=False)
            )
            self.fc_mu    = nn.Linear(latent_dim * 2, latent_dim)
            self.fc_logvar = nn.Linear(latent_dim * 2, latent_dim)
        else:
            self.use_skips = True
            self.layers = nn.ModuleList()
            # First layer.
            self.layers.append(
                nn.Sequential(
                    nn.Linear(input_dim, hidden_dims[0]),
                    nn.ReLU(inplace=False)
                )
            )
            # Subsequent residual blocks.
            for i in range(1, len(hidden_dims)):
                self.layers.append(ResidualBlock(hidden_dims[i-1], hidden_dims[i]))
            self.fc_mu    = nn.Linear(hidden_dims[-1], latent_dim)
            self.fc_logvar = nn.Linear(hidden_dims[-1], latent_dim)
            
    def forward(self, x):
        if not self.use_skips:
            x = self.model(x)
            return self.fc_mu(x), self.fc_logvar(x), None
        else:
            skip_connections = []
            for layer in self.layers:
                x = layer(x)
                skip_connections.append(x)
            mu = self.fc_mu(x)
            logvar = self.fc_logvar(x)
            return mu, logvar, skip_connections

In [28]:
encoder = Encoder(1024, 32, [512, 256, 128, 64])

In [33]:
mu, logvar, skip_connections = encoder.forward(first_layer)
mu.size()

torch.Size([1024, 32])

In [40]:
skip_connections[].size()

torch.Size([1024, 128])